# Identity-level risk control (ILRC) on MEVID-SC: no LLM, CPU only

Turns each backbone's fingerprints into **one global ID per person** across cameras, with a **guarantee on the final IDs**, not on single links.

- **Light verifier** (no model, milliseconds): cosine similarity, k-reciprocal re-ranking (Zhong et al., CVPR 2017), mutual rank, identity margin, and group similarity, combined by a small logistic regression fitted on the calibration group.
- **Global IDs:** average-linkage merging of clips.
- **ILRC (proposed):** chooses how far to merge using a **fixed-sequence test with a Clopper-Pearson bound, with each person as one unit**. Promise: with 90% confidence, **at most 10% of new people have more than 5% wrong links** in their ID. Anything not merged stays a separate, unresolved ID.
- **Protocol:** calibrate on one group of 27 people, freeze, test on the other 27, swap. Settings are frozen in `SETTINGS_frozen.json` before any result.

Compared on **CSCI** and **the OSNet baseline (OSNet)**: (a) similarity + point threshold (baseline), (b) light verifier + point threshold, (c) similarity + ILRC, (d) light verifier + ILRC (proposed).

Runs on a normal CPU runtime, about 5 to 15 minutes.

In [ ]:
# 1. Load both backbones' features on MEVID-SC + the frozen person groups
import os, glob, json, time, numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')

SD = '/content/drive/Shareddrives/*'
def find(pats):
    for p in pats:
        h = sorted(glob.glob(p, recursive=True))
        if h:
            return os.path.dirname(h[0])
    raise FileNotFoundError(pats[0])

CS_DIR = find([f'{SD}/Data/MEVID-results/CSCI-SameClothes/'
               'test_templates_csci_sc.npz',
               f'{SD}/**/CSCI-SameClothes/test_templates_csci_sc.npz'])
TR_DIR = find([f'{SD}/Data/MEVID-results/Tracker-SameClothes/'
               'test_templates_tracker_sc.npz',
               f'{SD}/**/Tracker-SameClothes/test_templates_tracker_sc.npz'])
OUT = os.path.join(os.path.dirname(CS_DIR), 'ILRC-SameClothes')
os.makedirs(OUT, exist_ok=True)

z = np.load(f'{CS_DIR}/test_templates_csci_sc.npz')
pid, cam = z['pid'], z['cam']
zt = np.load(f'{TR_DIR}/test_templates_tracker_sc.npz')
assert (zt['pid'] == pid).all() and (zt['cam'] == cam).all()
BACKBONES = {'CSCI': z['templates'], 'OSNet baseline (OSNet)': zt['zs']}
for k, v in BACKBONES.items():
    BACKBONES[k] = v / np.maximum(np.linalg.norm(v, axis=1,
                                                 keepdims=True), 1e-8)
proto = json.load(open(f'{CS_DIR}/PROTOCOL_frozen_sc.json'))
fold = np.array([proto['folds'][str(int(p))] for p in pid])

# frozen settings (fixed before any result of this notebook)
SETTINGS = {'pair_precision_target': 0.95,       # baseline rule
            'person_error_limit': 0.05,           # >5% wrong links = fail
            'epsilon': 0.10,                      # max share of people
            'delta': 0.10,                        # failing, 90% confidence
            'k1': 20, 'k2': 6, 'lambda': 0.3,     # k-reciprocal
            'margin_topk': 50, 'group_k': 5, 'seed': 2027}
with open(f'{OUT}/SETTINGS_frozen.json', 'w') as f:
    json.dump(SETTINGS, f, indent=2)
print('Backbones:', list(BACKBONES), '| clips:', len(pid),
      '| people per group:',
      {g: len(np.unique(pid[fold == g])) for g in 'AB'})
print('Results go to', OUT)


In [ ]:
# 2. The light verifier: three checks on the similarity table, no model
def similarity(T):
    S = np.clip(T @ T.T, -1, 1)
    return S


def kreciprocal_jaccard(S, k1, k2, lam):
    """k-reciprocal re-ranking (Zhong et al., CVPR 2017)."""
    n = len(S)
    D = 2 - 2 * S
    D = D / D.max()
    rank = np.argsort(D, axis=1)
    V = np.zeros((n, n), np.float32)
    h = int(round(k1 / 2))
    for i in range(n):
        fwd = rank[i, :k1 + 1]
        bwd = rank[fwd, :k1 + 1]
        kr = fwd[np.where(bwd == i)[0]]
        exp = kr
        for c in kr:
            cf = rank[c, :h + 1]
            cb = rank[cf, :h + 1]
            ckr = cf[np.where(cb == c)[0]]
            if len(np.intersect1d(ckr, kr)) > 2 / 3 * len(ckr):
                exp = np.append(exp, ckr)
        exp = np.unique(exp)
        w = np.exp(-D[i, exp])
        V[i, exp] = w / w.sum()
    if k2 > 1:
        V = np.stack([V[rank[i, :k2]].mean(0) for i in range(n)])
    J = np.zeros((n, n), np.float32)
    for i in range(n):
        nz = np.flatnonzero(V[i])
        m = np.minimum(V[i, nz][None, :], V[:, nz]).sum(1)
        J[i] = 1 - m / (2 - m)
    J = (J + J.T) / 2
    return (1 - lam) * J + lam * D


def identity_margin(S, tau, topk):
    """s(i,j) minus the best similarity of i to a clip NOT linked to j
    (linked = similarity to j >= tau): the identity-level margin."""
    n = len(S)
    M = np.zeros((n, n), np.float32)
    for i in range(n):
        order = np.argsort(-S[i])
        cand = order[order != i][:topk]
        free = S[cand] < tau                     # topk x n: unlinked to j
        first = np.where(free.any(0), free.argmax(0), topk - 1)
        M[i] = S[i] - S[i, cand[first]]
    return (M + M.T) / 2


def group_similarity(S, k):
    """mean similarity of i to j's k nearest clips (set-to-set)."""
    nn = np.argsort(-S, axis=1)[:, :k]
    G = np.zeros_like(S)
    for j in range(len(S)):
        G[:, j] = S[:, nn[j]].mean(1)
    return (G + G.T) / 2


def features(T):
    S = similarity(T)
    st = SETTINGS
    R = np.argsort(np.argsort(-S, axis=1), axis=1)
    feats = {'similarity': S,
             'reranked distance': kreciprocal_jaccard(
                 S, st['k1'], st['k2'], st['lambda']),
             'mutual rank (log)': np.log1p(np.maximum(R, R.T)),
             'group similarity': group_similarity(S, st['group_k'])}
    return feats


def add_margin(feats, tau):
    feats['identity margin'] = identity_margin(
        feats['similarity'], tau, SETTINGS['margin_topk'])
    return feats


def pair_table(feats, idx):
    iu = np.triu_indices(len(idx), 1)
    return np.stack([F[np.ix_(idx, idx)][iu] for F in feats.values()], 1)


In [ ]:
# 3. Global IDs + risk control, calibrated on one group, tested on the
#    other (then swapped). Four methods per backbone:
#    (a) similarity + point threshold  (baseline, like step 3)
#    (b) light verifier + point threshold
#    (c) similarity + identity-level risk control (ILRC)
#    (d) light verifier + ILRC          (proposed)
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.optimize import linear_sum_assignment
from scipy.stats import beta
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline


def person_losses(pred, p):
    """per person: share of its same-ID pairs that are other people."""
    out = {}
    same_c = pred[:, None] == pred[None, :]
    np.fill_diagonal(same_c, False)
    for q in np.unique(p):
        rows = same_c[p == q]
        tot = rows.sum()
        wrong = rows[:, p != q].sum()
        out[q] = wrong / tot if tot else 0.0
    return out


def score(pred, p, c):
    P = {x: i for i, x in enumerate(np.unique(p))}
    C = {x: j for j, x in enumerate(np.unique(pred))}
    M = np.zeros((len(P), len(C)))
    for a, b in zip(p, pred):
        M[P[a], C[b]] += 1
    r, cc = linear_sum_assignment(-M)
    sp = p[:, None] == p[None, :]; sc = pred[:, None] == pred[None, :]
    off = ~np.eye(len(p), dtype=bool); tp = (sp & sc & off).sum()
    xc = sp & (c[:, None] != c[None, :])
    lp = np.array(list(person_losses(pred, p).values()))
    return {'IDF1 %': 100 * M[r, cc].sum() / len(p),
            'IDs per person': (M > 0).sum(1).mean(),
            'people per ID': (M > 0).sum(0).mean(),
            'pair precision %': (100 * tp / (sc & off).sum()
                                 if (sc & off).sum() else np.nan),
            'cross-camera recall %': 100 * (xc & sc).sum() / max(1, xc.sum()),
            'people failing %':
                100 * (lp > SETTINGS['person_error_limit']).mean()}


def cluster(D):
    return linkage(D[np.triu_indices(len(D), 1)], 'average')


def labels(Z, thr):
    return fcluster(Z, thr, 'distance')


def choose_point(Z, p, c, grid):
    """baseline: most merging threshold with pair precision >= target."""
    best = grid[0]
    for d in grid:                                # strict -> permissive
        if score(labels(Z, d), p, c)['pair precision %'] >= \
                100 * SETTINGS['pair_precision_target']:
            best = d
    return best


def choose_ilrc(Z, p, grid):
    """ILRC: fixed-sequence test from strict to permissive; accept a
    threshold while the Clopper-Pearson upper bound on the share of
    people with > 5% wrong links stays <= epsilon (90% confidence).
    Stops at the first failure, which keeps the guarantee valid."""
    st, best = SETTINGS, None
    for d in grid:
        lp = np.array(list(person_losses(labels(Z, d), p).values()))
        k, n = int((lp > st['person_error_limit']).sum()), len(lp)
        ucb = beta.ppf(1 - st['delta'], k + 1, n - k) if k < n else 1.0
        if ucb > st['epsilon']:
            break
        best = d
    return best


GRID = np.round(np.arange(0.01, 0.951, 0.01), 2)   # merge distances
rows, used = [], {}
for bb, T in BACKBONES.items():
    t0 = time.time()
    # features are computed inside each group separately (closed world)
    FE = {g: features(T[fold == g]) for g in 'AB'}
    for cal, ev in [('A', 'B'), ('B', 'A')]:
        ic, ie = np.flatnonzero(fold == cal), np.flatnonzero(fold == ev)
        Sc, Se = FE[cal]['similarity'], FE[ev]['similarity']
        # link threshold for the identity margin: 95th percentile of
        # different-person similarity in the calibration group
        diff = pid[ic][:, None] != pid[ic][None, :]
        tau = float(np.quantile(Sc[diff], 0.95))
        f2c = add_margin(dict(FE[cal]), tau)
        f2e = add_margin(dict(FE[ev]), tau)
        Xc = pair_table(f2c, np.arange(len(ic)))
        Xe = pair_table(f2e, np.arange(len(ie)))
        iu_c = np.triu_indices(len(ic), 1)
        yc = (pid[ic][:, None] == pid[ic][None, :])[iu_c]
        lr = make_pipeline(StandardScaler(),
                           LogisticRegression(max_iter=2000))
        lr.fit(Xc, yc)

        def prob_matrix(X, n):
            Pm = np.zeros((n, n))
            Pm[np.triu_indices(n, 1)] = lr.predict_proba(X)[:, 1]
            Pm = Pm + Pm.T
            np.fill_diagonal(Pm, 1)
            return Pm

        dist = {'similarity': (1 - Sc, 1 - Se),
                'light verifier': (1 - prob_matrix(Xc, len(ic)),
                                   1 - prob_matrix(Xe, len(ie)))}
        for score_name, (Dc, De) in dist.items():
            Zc, Ze = cluster(Dc), cluster(De)
            for rule in ['point threshold', 'ILRC']:
                if rule == 'point threshold':
                    d = choose_point(Zc, pid[ic], cam[ic], GRID)
                else:
                    d = choose_ilrc(Zc, pid[ic], GRID)
                if d is None:                    # nothing passes: no merges
                    pc, pe = np.arange(len(ic)), np.arange(len(ie))
                else:
                    pc, pe = labels(Zc, d), labels(Ze, d)
                cal_s = score(pc, pid[ic], cam[ic])
                ev_s = score(pe, pid[ie], cam[ie])
                rows.append({'backbone': bb, 'score': score_name,
                             'rule': rule,
                             'calibrate->evaluate': f'{cal}->{ev}',
                             'merge if distance <=': d,
                             'promised pair precision %':
                                 round(cal_s['pair precision %'], 1),
                             **{k: round(v, 2) for k, v in ev_s.items()}})
        used[f'{bb} {cal}->{ev}'] = {'tau': round(tau, 4), 'weights': dict(
            zip(f2c.keys(), np.round(lr[-1].coef_[0], 3).tolist()))}
    print(f'{bb}: done in {time.time() - t0:.0f} s')

res = pd.DataFrame(rows)
res.to_csv(f'{OUT}/ilrc_results_by_direction.csv', index=False)
with open(f'{OUT}/verifier_weights_and_tau.json', 'w') as f:
    json.dump(used, f, indent=2)
print('Saved per-direction results to', OUT)


In [ ]:
# 4. Results: mean of both directions, per backbone and method
cols = ['IDF1 %', 'IDs per person', 'people per ID', 'pair precision %',
        'promised pair precision %', 'cross-camera recall %',
        'people failing %']
summary = (res.groupby(['backbone', 'score', 'rule'], sort=False)[cols]
           .mean().round(1))
summary.to_csv(f'{OUT}/ilrc_summary.csv')
print('CONSISTENT ID ON MEVID-SC (held-out group; mean of A->B and B->A)')
print('ILRC promise: at most', int(100 * SETTINGS['epsilon']),
      '% of new people have more than',
      int(100 * SETTINGS['person_error_limit']),
      '% wrong links (90% confidence)')
display(summary)
print('\nPer direction (check the promise held in BOTH):')
display(res[['backbone', 'score', 'rule', 'calibrate->evaluate',
             'IDF1 %', 'IDs per person', 'pair precision %',
             'promised pair precision %', 'people failing %']])
